# MT Model Training with LoRA + BF16 - Sequential Fine-Tuning Experiments

This notebook tests whether **sequential fine-tuning with LoRA adapters** (similar language → baseline) improves MT performance compared to **direct fine-tuning with LoRA** (baseline only).

## Why LoRA?
- **Memory Efficient:** Only trains low-rank adapter matrices instead of full model weights
- **Faster Training:** Fewer parameters to update = faster iterations
- **Same Hypothesis:** Test if similarity transfer helps when using parameter-efficient fine-tuning

## ⚡ BF16 Optimization
- **BF16 Mixed Precision:** ~1.5-2x speedup on RTX 3050 and newer GPUs
- **Better Stability:** BF16 has wider dynamic range than FP16
- **Native Support:** RTX 30/40 series have native BF16 Tensor Cores

## Experimental Design: Sequential Fine-Tuning with LoRA

For each target language, we create TWO models:

### 1. Baseline Models (Direct LoRA Training)
- **Start:** mBART-50 (pretrained)
- **Train:** Apply LoRA adapters and train Distant language → Target language (e.g., `en→tl`)
- **Result:** Baseline Model with LoRA adapters

### 2. Experimental Models (Sequential LoRA Fine-Tuning)
- **Start:** mBART-50 (pretrained)
- **Step 1:** Apply LoRA adapters and train Similar language → Target language (e.g., `bik→tl`)
- **Step 2:** Load Stage 1 LoRA adapters and **continue training** on Distant language → Target language (e.g., `en→tl`)
- **Result:** Experimental Model (with similarity transfer via LoRA)

### Three Target Languages
1. **Tagalog (tl)**
   - Baseline: `en→tl` only
   - Experimental: `bik→tl` THEN `en→tl`

2. **Ilonggo/Hiligaynon (hil)**
   - Baseline: `en→hil` only
   - Experimental: `msb→hil` THEN `en→hil`

3. **Waray (war)**
   - Baseline: `en→war` only
   - Experimental: `hil→war` THEN `en→war`

## Research Question
**Does "warming up" the model with LoRA on a similar low-resource language first improve performance on the baseline task?**

Expected: `BLEU(Experimental) > BLEU(Baseline)`

## Install Required Packages

Run this cell first if packages are not installed. Note the addition of `peft` for LoRA support.

In [ ]:
# Uncomment and run if needed
# !pip install transformers datasets evaluate sacrebleu torch sentencepiece accelerate peft

## Imports

In [ ]:
from transformers import (
    MBartForConditionalGeneration,
    MBartTokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import Dataset, DatasetDict
import evaluate
import numpy as np
import torch
from pathlib import Path
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

## Configuration

Set up training configurations for all language pairs, including LoRA-specific parameters and BF16 optimization.

In [ ]:
# Model configuration
MODEL_NAME = "facebook/mbart-large-50"
MAX_LENGTH = 128
BATCH_SIZE = 4  # Reduced for LoRA (still memory efficient)
LEARNING_RATE = 3e-4  # Higher LR often works well with LoRA
NUM_EPOCHS_STAGE1 = 3  # For similar language training (Stage 1)
NUM_EPOCHS_STAGE2 = 3  # For baseline training (Stage 2)

# ⚡ BF16 Optimization
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# LoRA configuration
LORA_R = 8  # Rank of LoRA matrices
LORA_ALPHA = 16  # Scaling factor
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # Apply LoRA to attention layers

# Directories
DATA_DIR = Path("../data/splits")
OUTPUT_DIR = Path("../models")
LOGS_DIR = Path("../logs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Experimental configurations
EXPERIMENTS = {
    "tagalog": {
        "target": "tl",
        "baseline_pair": "en-tl",
        "similar_pair": "bik-tl",
        "baseline_config": {
            "src_lang": "en_XX",
            "tgt_lang": "tl_XX",
        },
        "similar_config": {
            "src_lang": "tl_XX",
            "tgt_lang": "tl_XX",
        }
    },
    "ilonggo": {
        "target": "hil",
        "baseline_pair": "en-hil",
        "similar_pair": "msb-hil",
        "baseline_config": {
            "src_lang": "en_XX",
            "tgt_lang": "tl_XX",
        },
        "similar_config": {
            "src_lang": "tl_XX",
            "tgt_lang": "tl_XX",
        }
    },
    "waray": {
        "target": "war",
        "baseline_pair": "en-war",
        "similar_pair": "hil-war",
        "baseline_config": {
            "src_lang": "en_XX",
            "tgt_lang": "tl_XX",
        },
        "similar_config": {
            "src_lang": "tl_XX",
            "tgt_lang": "tl_XX",
        }
    }
}

print("Experimental Configuration (LoRA + BF16):")
print(f"  Model: {MODEL_NAME}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  ⚡ BF16 enabled: {USE_BF16}")
print(f"  LoRA rank (r): {LORA_R}")
print(f"  LoRA alpha: {LORA_ALPHA}")
print(f"  LoRA target modules: {LORA_TARGET_MODULES}")
print(f"  Epochs (Stage 1): {NUM_EPOCHS_STAGE1}")
print(f"  Epochs (Stage 2): {NUM_EPOCHS_STAGE2}")
print(f"\nTarget Languages: {len(EXPERIMENTS)}")
for lang, config in EXPERIMENTS.items():
    print(f"  - {lang.capitalize()}: {config['baseline_pair']} (baseline) vs {config['similar_pair']}→{config['baseline_pair']} (sequential)")

## Helper Functions

Functions to load data, preprocess, evaluate models, and apply LoRA adapters.

In [ ]:
def load_data_for_pair(pair_name):
    """Load train and dev splits for a language pair."""
    src_code, tgt_code = pair_name.split("-")
    pair_dir = DATA_DIR / pair_name
    
    with open(pair_dir / f"train.{src_code}", "r", encoding="utf-8") as f:
        train_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"train.{tgt_code}", "r", encoding="utf-8") as f:
        train_tgt = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"dev.{src_code}", "r", encoding="utf-8") as f:
        dev_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"dev.{tgt_code}", "r", encoding="utf-8") as f:
        dev_tgt = [line.strip() for line in f.readlines()]
    
    train_dataset = Dataset.from_dict({"src": train_src, "tgt": train_tgt})
    dev_dataset = Dataset.from_dict({"src": dev_src, "tgt": dev_tgt})
    
    dataset_dict = DatasetDict({"train": train_dataset, "validation": dev_dataset})
    
    print(f"Loaded {pair_name}: Train={len(train_dataset)}, Dev={len(dev_dataset)}")
    return dataset_dict


def create_preprocess_function(tokenizer, src_lang, tgt_lang, max_length):
    """Create preprocessing function for tokenization."""
    def preprocess(batch):
        tokenizer.src_lang = src_lang
        inputs = tokenizer(
            batch["src"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )
        tokenizer.tgt_lang = tgt_lang
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                batch["tgt"],
                truncation=True,
                padding="max_length",
                max_length=max_length
            )
        inputs["labels"] = labels["input_ids"]
        return inputs
    return preprocess


def create_compute_metrics(tokenizer):
    """Create function to compute BLEU score during evaluation."""
    bleu = evaluate.load("sacrebleu")
    
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        if isinstance(preds, tuple):
            preds = preds[0]
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        result = bleu.compute(
            predictions=decoded_preds,
            references=[[label] for label in decoded_labels]
        )
        return {"bleu": result["score"]}
    return compute_metrics


def apply_lora_to_model(model):
    """Apply LoRA adapters to the model."""
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="SEQ_2_SEQ_LM",
    )
    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()
    return peft_model


print("✓ Helper functions defined")

## Training Functions

Functions to train single stages and run complete experiments with LoRA and BF16.

In [ ]:
def train_single_stage(pair_name, config, model_name_or_path, output_subdir, num_epochs, stage_name="", is_stage2=False):
    """Train a single stage with LoRA adapters and BF16."""
    print("\n" + "="*80)
    print(f"Training: {pair_name.upper()}")
    if stage_name:
        print(f"Stage: {stage_name}")
    print("="*80)
    
    # Load tokenizer
    print("\n1. Loading tokenizer...")
    tokenizer = MBartTokenizerFast.from_pretrained(MODEL_NAME)
    
    # Load model
    print("\n2. Loading model...")
    if is_stage2:
        # Stage 2: Load base model then load Stage 1 adapters
        print(f"   Loading Stage 1 adapters from: {model_name_or_path}")
        base_model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
        model = PeftModel.from_pretrained(base_model, model_name_or_path, is_trainable=True)
        
        # Explicitly enable training mode and mark adapters as trainable
        model.train()
        model.base_model.train()
        
        # Print trainable parameters to verify
        model.print_trainable_parameters()
        
        print("   Stage 1 adapters loaded successfully")
        print("   Model configured for continued training")
    else:
        # Stage 1 or Baseline: Load base model and apply new LoRA
        print(f"   Loading base model: {MODEL_NAME}")
        base_model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
        model = apply_lora_to_model(base_model)
    
    # Load dataset
    print("\n3. Loading dataset...")
    dataset = load_data_for_pair(pair_name)
    
    # Tokenize
    print("\n4. Tokenizing dataset...")
    preprocess_fn = create_preprocess_function(
        tokenizer, config['src_lang'], config['tgt_lang'], MAX_LENGTH
    )
    tokenized_dataset = dataset.map(preprocess_fn, batched=True)
    
    # Training arguments with BF16
    output_dir = OUTPUT_DIR / output_subdir
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=num_epochs,
        save_strategy="epoch",
        save_total_limit=2,
        predict_with_generate=True,
        logging_dir=str(LOGS_DIR / output_subdir),
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        bf16=USE_BF16,  # ⚡ BF16 optimization
        fp16=False,  # Disable FP16 when using BF16
        report_to="none",
    )
    
    # Create trainer
    print("\n5. Setting up trainer...")
    print(f"   Using BF16: {USE_BF16}")
    compute_metrics_fn = create_compute_metrics(tokenizer)
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_fn,
    )
    
    # Train
    print("\n6. Starting training...")
    train_result = trainer.train()
    
    # Save model (LoRA adapters)
    print("\n7. Saving LoRA adapters and tokenizer...")
    final_model_path = output_dir / "final_model"
    model.save_pretrained(str(final_model_path))
    tokenizer.save_pretrained(str(final_model_path))
    
    # Evaluate
    print("\n8. Final evaluation...")
    eval_results = trainer.evaluate()
    
    # Save results
    results = {
        "pair": pair_name,
        "stage": stage_name,
        "model_source": model_name_or_path,
        "bf16_enabled": USE_BF16,
        "lora_config": {
            "r": LORA_R,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
            "target_modules": LORA_TARGET_MODULES
        },
        "train_results": {
            "train_loss": train_result.training_loss,
            "train_runtime": train_result.metrics["train_runtime"],
        },
        "eval_results": eval_results
    }
    
    results_file = output_dir / "training_results.json"
    with open(results_file, "w") as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✓ Training complete")
    print(f"  Final BLEU: {eval_results['eval_bleu']:.2f}")
    print(f"  Training time: {train_result.metrics['train_runtime']:.1f}s")
    print(f"  LoRA adapters saved to: {final_model_path}")
    
    return results, str(final_model_path)


def train_experiment(target_lang_name, experiment_config):
    """Run complete experiment for one target language with LoRA and BF16."""
    print("\n" + "#"*80)
    print(f"# EXPERIMENT (LoRA + BF16): {target_lang_name.upper()}")
    print("#"*80)
    
    baseline_pair = experiment_config['baseline_pair']
    similar_pair = experiment_config['similar_pair']
    all_results = {}
    
    # BASELINE: Direct LoRA training on distant → target
    print(f"\n{'='*80}")
    print(f"BASELINE (LoRA + BF16): {baseline_pair}")
    print("="*80)
    baseline_results, baseline_model_path = train_single_stage(
        pair_name=baseline_pair,
        config=experiment_config['baseline_config'],
        model_name_or_path=MODEL_NAME,
        output_subdir=f"{target_lang_name}_baseline_lora_bf16",
        num_epochs=NUM_EPOCHS_STAGE2,
        stage_name="Baseline (Direct LoRA + BF16)",
        is_stage2=False
    )
    all_results['baseline'] = baseline_results
    
    # EXPERIMENTAL - STAGE 1: LoRA on similar → target
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL (LoRA + BF16) - STAGE 1: {similar_pair}")
    print("="*80)
    stage1_results, stage1_model_path = train_single_stage(
        pair_name=similar_pair,
        config=experiment_config['similar_config'],
        model_name_or_path=MODEL_NAME,
        output_subdir=f"{target_lang_name}_experimental_stage1_lora_bf16",
        num_epochs=NUM_EPOCHS_STAGE1,
        stage_name="Stage 1 (Similar Language LoRA + BF16)",
        is_stage2=False
    )
    all_results['experimental_stage1'] = stage1_results
    
    # EXPERIMENTAL - STAGE 2: Continue LoRA from Stage 1 on distant → target
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL (LoRA + BF16) - STAGE 2: {baseline_pair}")
    print(f"Continuing from Stage 1 LoRA adapters")
    print("="*80)
    stage2_results, stage2_model_path = train_single_stage(
        pair_name=baseline_pair,
        config=experiment_config['baseline_config'],
        model_name_or_path=stage1_model_path,
        output_subdir=f"{target_lang_name}_experimental_stage2_lora_bf16",
        num_epochs=NUM_EPOCHS_STAGE2,
        stage_name="Stage 2 (Baseline after Similar LoRA + BF16)",
        is_stage2=True
    )
    all_results['experimental_stage2'] = stage2_results
    
    # SUMMARY
    print("\n" + "="*80)
    print(f"EXPERIMENT COMPLETE (LoRA + BF16): {target_lang_name.upper()}")
    print("="*80)
    print(f"\nBaseline (LoRA + BF16): {all_results['baseline']['eval_results']['eval_bleu']:.2f} BLEU")
    print(f"Experimental Stage 1: {all_results['experimental_stage1']['eval_results']['eval_bleu']:.2f} BLEU")
    print(f"Experimental Stage 2: {all_results['experimental_stage2']['eval_results']['eval_bleu']:.2f} BLEU")
    
    improvement = all_results['experimental_stage2']['eval_results']['eval_bleu'] - all_results['baseline']['eval_results']['eval_bleu']
    print(f"\nImprovement: {improvement:+.2f} BLEU points")
    if improvement > 0:
        print("✓ Sequential LoRA fine-tuning IMPROVED performance")
    elif improvement < 0:
        print("✗ Sequential LoRA fine-tuning DEGRADED performance")
    else:
        print("= No difference in performance")
    
    # Save summary
    summary_file = OUTPUT_DIR / f"{target_lang_name}_lora_bf16_experiment_summary.json"
    with open(summary_file, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSummary saved to: {summary_file}")
    
    return all_results


print("✓ Training functions defined")

## Run Single Experiment (LoRA + BF16)

Test with one target language first to verify the LoRA + BF16 setup.

In [ ]:
# Run one experiment (uncomment to test)
# target_lang = "tagalog"
# results = train_experiment(target_lang, EXPERIMENTS[target_lang])

## Run Individual Stages (Advanced)

If you need to run only specific stages (e.g., if Baseline and Stage 1 are done), use these cells:

In [ ]:
# Run only BASELINE (skip if already done)
# target_lang = "tagalog"
# exp_config = EXPERIMENTS[target_lang]
# baseline_results, baseline_model_path = train_single_stage(
#    pair_name=exp_config['baseline_pair'],
#    config=exp_config['baseline_config'],
#    model_name_or_path=MODEL_NAME,
#    output_subdir=f"{target_lang}_baseline_lora_bf16",
#    num_epochs=NUM_EPOCHS_STAGE2,
#    stage_name="Baseline (Direct LoRA + BF16)",
#    is_stage2=False
#)


In [14]:
# Run only STAGE 1 (skip if already done)
target_lang = "tagalog"
exp_config = EXPERIMENTS[target_lang]
stage1_results, stage1_model_path = train_single_stage(
    pair_name=exp_config['similar_pair'],
    config=exp_config['similar_config'],
    model_name_or_path=MODEL_NAME,
    output_subdir=f"{target_lang}_experimental_stage1_lora_bf16",
    num_epochs=NUM_EPOCHS_STAGE1,
    stage_name="Stage 1 (Similar Language LoRA + BF16)",
   is_stage2=False
)


Training: BIK-TL
Stage: Stage 1 (Similar Language LoRA + BF16)

1. Loading tokenizer...


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizerFast'.



2. Loading model...
   Loading base model: facebook/mbart-large-50
trainable params: 1,179,648 || all params: 612,059,136 || trainable%: 0.1927

3. Loading dataset...
Loaded bik-tl: Train=2358, Dev=294

4. Tokenizing dataset...


Map:   0%|          | 0/2358 [00:00<?, ? examples/s]

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


...

C:\Users\user\AppData\Local\Temp\ipykernel_11352\1374146748.py:73: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(



6. Starting training...


Epoch,Training Loss,Validation Loss,Bleu
1,8.519000,7.878166,2.806731
2,7.846100,7.771974,3.379977
3,7.778400,7.754771,4.134649



7. Saving LoRA adapters and tokenizer...

8. Final evaluation...


KeyboardInterrupt: 

In [ ]:
# Run only STAGE 2 (continuing from Stage 1)
# ⚠️ Make sure Stage 1 model exists first!
target_lang = "tagalog"
exp_config = EXPERIMENTS[target_lang]

# Path to Stage 1 model (adjust if needed)
stage1_model_path = OUTPUT_DIR / f"{target_lang}_experimental_stage1_lora_bf16" / "final_model"

stage2_results, stage2_model_path = train_single_stage(
    pair_name=exp_config['baseline_pair'],
    config=exp_config['baseline_config'],
    model_name_or_path=str(stage1_model_path),
    output_subdir=f"{target_lang}_experimental_stage2_lora_bf16",
    num_epochs=NUM_EPOCHS_STAGE2,
    stage_name="Stage 2 (Baseline after Similar LoRA + BF16)",
    is_stage2=True
)

print(f"\n✓ Stage 2 complete!")
print(f"  BLEU: {stage2_results['eval_results']['eval_bleu']:.2f}")

## Run All Experiments (LoRA + BF16)

Run all three experiments with LoRA adapters and BF16 optimization. **WARNING: This will take several hours!**

For each target language:
1. Train baseline LoRA model (distant → target)
2. Train Stage 1 LoRA (similar → target)
3. Train Stage 2 LoRA (continue with distant → target)

Total: 9 training runs (3 per target language × 3 target languages)

**Expected speedup with BF16: ~1.5-2x faster than FP16/FP32**

In [ ]:
# Run all experiments (uncomment to run)
# all_experiment_results = {}
#
# for target_lang, exp_config in EXPERIMENTS.items():
#     try:
#         results = train_experiment(target_lang, exp_config)
#         all_experiment_results[target_lang] = results
#     except Exception as e:
#         print(f"\n❌ Error in {target_lang} experiment: {e}")
#         import traceback
#         traceback.print_exc()
#         continue
#
# # Save overall summary
# final_summary_file = OUTPUT_DIR / "all_lora_bf16_experiments_summary.json"
# with open(final_summary_file, "w") as f:
#     json.dump(all_experiment_results, f, indent=2)
#
# print("\n" + "#"*80)
# print("# ALL LoRA + BF16 EXPERIMENTS COMPLETE")
# print("#"*80)
# print(f"\nFinal summary saved to: {final_summary_file}")
#
# # Print comparison table
# print("\n" + "="*80)
# print("RESULTS SUMMARY (LoRA + BF16)")
# print("="*80)
# print(f"{'Target':<20} {'Baseline BLEU':<15} {'Sequential BLEU':<15} {'Improvement':<15}")
# print("-"*80)
# for target_lang, results in all_experiment_results.items():
#     baseline_bleu = results['baseline']['eval_results']['eval_bleu']
#     sequential_bleu = results['experimental_stage2']['eval_results']['eval_bleu']
#     improvement = sequential_bleu - baseline_bleu
#     print(f"{target_lang.capitalize():<20} {baseline_bleu:<15.2f} {sequential_bleu:<15.2f} {improvement:+.2f}")

## Summary and Next Steps

### What This Notebook Does

This notebook implements **sequential fine-tuning experiments with LoRA adapters and BF16 mixed precision** to test whether "warming up" a model on similar language data improves performance on the baseline task while using parameter-efficient fine-tuning with accelerated training.

### Key Differences from Base LoRA Notebook

1. **BF16 Mixed Precision**: ~1.5-2x faster training on RTX 3050 and newer GPUs
2. **Better Numerical Stability**: BF16 has wider dynamic range than FP16
3. **Native Hardware Support**: Utilizes BF16 Tensor Cores on RTX 30/40 series
4. **Same Memory Usage**: BF16 uses same memory as FP16 but with better stability

### BF16 Benefits

- ⚡ **Faster Training**: 1.5-2x speedup compared to FP16/FP32
- 🎯 **Better Stability**: Wider dynamic range reduces overflow/underflow issues
- 💾 **Same Memory**: Uses 16 bits like FP16 but allocates differently
- 🔧 **Hardware Accelerated**: Native support on Ampere (RTX 30) and newer

### Output Structure
```
models/
  ├── tagalog_baseline_lora_bf16/
  │   └── final_model/          # Baseline: en→tl (LoRA + BF16)
  ├── tagalog_experimental_stage1_lora_bf16/
  │   └── final_model/          # Stage 1: bik→tl (LoRA + BF16)
  ├── tagalog_experimental_stage2_lora_bf16/
  │   └── final_model/          # Stage 2: bik→tl THEN en→tl (LoRA + BF16)
  ├── tagalog_lora_bf16_experiment_summary.json
  │
  ├── ilonggo_baseline_lora_bf16/
  ├── ilonggo_experimental_stage1_lora_bf16/
  ├── ilonggo_experimental_stage2_lora_bf16/
  ├── ilonggo_lora_bf16_experiment_summary.json
  │
  ├── waray_baseline_lora_bf16/
  ├── waray_experimental_stage1_lora_bf16/
  ├── waray_experimental_stage2_lora_bf16/
  ├── waray_lora_bf16_experiment_summary.json
  │
  └── all_lora_bf16_experiments_summary.json
```

### Comparison with Other Notebooks

After running all notebooks, you can compare:
1. **Full fine-tuning** (`train-models.ipynb`)
2. **LoRA (FP16/FP32)** (`train-models-lora.ipynb`)
3. **LoRA + BF16** (this notebook)
4. **Full RTX 3050 optimizations** (`train-models-lora-rtx3050.ipynb`)

### Expected Analysis

For each target language:
- Compare `BLEU(Baseline LoRA + BF16)` vs. `BLEU(Experimental LoRA + BF16)`
- Compare training time vs. base LoRA notebook (~1.5-2x faster expected)
- Calculate improvement: `Δ BLEU = Experimental - Baseline`

### Training Notes

- **Total Training Runs:** 9 (3 per target language)
- **GPU Recommended:** RTX 3050 or newer (with BF16 support)
- **Time Estimate:** ~20-30 min per run = 3-4.5 hours total (vs 4-7 hours with FP16)
- **Memory:** Same as base LoRA (~2-4GB VRAM)
- **Speedup:** ~1.5-2x faster than FP16/FP32

### Tips

1. Check BF16 support: `torch.cuda.is_bf16_supported()`
2. Monitor GPU usage with `nvidia-smi`
3. Compare training times with base LoRA notebook
4. BF16 may show slightly different loss values (but similar convergence)
5. If BF16 not supported, notebook will automatically fall back to FP16/FP32

## Calculate Improvement

Compare Baseline vs Experimental (Stage 2) results to see if similarity transfer helped.

In [ ]:
# Calculate improvement after running Baseline and Stage 2
import json

target_lang = "tagalog"

# Load results
baseline_results_file = OUTPUT_DIR / f"{target_lang}_baseline_lora_bf16" / "training_results.json"
stage2_results_file = OUTPUT_DIR / f"{target_lang}_experimental_stage2_lora_bf16" / "training_results.json"

if baseline_results_file.exists() and stage2_results_file.exists():
    # Load results
    with open(baseline_results_file, 'r') as f:
        baseline_results = json.load(f)
    
    with open(stage2_results_file, 'r') as f:
        stage2_results = json.load(f)
    
    # Extract BLEU scores
    baseline_bleu = baseline_results['eval_results']['eval_bleu']
    stage2_bleu = stage2_results['eval_results']['eval_bleu']
    
    # Calculate improvement
    improvement = stage2_bleu - baseline_bleu
    improvement_pct = (improvement / baseline_bleu) * 100
    
    # Display results
    print("="*70)
    print(f"  RESULTS COMPARISON: {target_lang.upper()}")
    print("="*70)
    print()
    print(f"📊 Baseline (en→tl only):                 {baseline_bleu:.2f} BLEU")
    print(f"🔄 Experimental (bik→tl THEN en→tl):      {stage2_bleu:.2f} BLEU")
    print()
    print("-"*70)
    print(f"📈 Improvement:                            {improvement:+.2f} BLEU points")
    print(f"   Relative improvement:                  {improvement_pct:+.2f}%")
    print("-"*70)
    print()
    
    if improvement > 0:
        print("✅ RESULT: Sequential fine-tuning (similarity transfer) IMPROVED performance!")
        print(f"   → Warming up with similar language (bik→tl) helped!")
    elif improvement < -0.5:
        print("❌ RESULT: Sequential fine-tuning DEGRADED performance significantly")
        print(f"   → Similar language transfer may have hurt learning")
    elif improvement < 0:
        print("⚠️  RESULT: Sequential fine-tuning slightly degraded performance")
        print(f"   → Small negative effect, possibly within noise")
    else:
        print("➖ RESULT: No significant difference between approaches")
        print(f"   → Similar language transfer had neutral effect")
    
    print()
    print("="*70)
    
    # Training time comparison
    baseline_time = baseline_results['train_results']['train_runtime']
    stage1_results_file = OUTPUT_DIR / f"{target_lang}_experimental_stage1_lora_bf16" / "training_results.json"
    
    if stage1_results_file.exists():
        with open(stage1_results_file, 'r') as f:
            stage1_results = json.load(f)
        stage1_time = stage1_results['train_results']['train_runtime']
        stage2_time = stage2_results['train_results']['train_runtime']
        total_experimental_time = stage1_time + stage2_time
        
        print(f"\n⏱️  TRAINING TIME:")
        print(f"   Baseline:     {baseline_time/60:.1f} min")
        print(f"   Stage 1:      {stage1_time/60:.1f} min")
        print(f"   Stage 2:      {stage2_time/60:.1f} min")
        print(f"   Experimental: {total_experimental_time/60:.1f} min (total)")
        print(f"   Overhead:     {(total_experimental_time - baseline_time)/60:+.1f} min ({((total_experimental_time/baseline_time - 1)*100):+.1f}%)")
        print("="*70)

else:
    print("⚠️  Results not found. Make sure you've run:")
    print("   1. Baseline training")
    print("   2. Stage 1 training")
    print("   3. Stage 2 training")
    print()
    print("Missing files:")
    if not baseline_results_file.exists():
        print(f"   ❌ {baseline_results_file}")
    if not stage2_results_file.exists():
        print(f"   ❌ {stage2_results_file}")

## GPU Cache Cleanup

In [13]:
import gc
import torch

print("🧹 Cleaning up memory...")

# Clear Python garbage collector
gc.collect()

# Clear PyTorch CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    
    # Get current GPU memory stats
    allocated = torch.cuda.memory_allocated() / 1024**3  # Convert to GB
    reserved = torch.cuda.memory_reserved() / 1024**3
    
    print(f"✓ CUDA cache cleared")
    print(f"  GPU Memory Allocated: {allocated:.2f} GB")
    print(f"  GPU Memory Reserved:  {reserved:.2f} GB")
else:
    print("✓ Garbage collection complete (No GPU available)")

print("\n💾 Memory cleanup complete!")

🧹 Cleaning up memory...
✓ CUDA cache cleared
  GPU Memory Allocated: 0.02 GB
  GPU Memory Reserved:  0.50 GB

💾 Memory cleanup complete!
✓ CUDA cache cleared
  GPU Memory Allocated: 0.02 GB
  GPU Memory Reserved:  0.50 GB

💾 Memory cleanup complete!
